In [1]:
import pandas as pd

In [2]:
'''
- group by
- sort_by
- rename
- rename_axis
- to_markdown
- map
- aggregate functions -> sum, count, mean, max, min, nunique(count of unique numbers)

dataframe.groupby('col_name')[col_2].agg_function()
dataframe.groupby(['col_1',col_2])[col_3].agg_function()
'''

"\n- group by\n- sort_by\n- rename\n- rename_axis\n- to_markdown\n- map\n- aggregate functions -> sum, count, mean, max, min, nunique(count of unique numbers)\n\ndataframe.groupby('col_name')[col_2].agg_function()\ndataframe.groupby(['col_1',col_2])[col_3].agg_function()\n"

In [3]:
df=pd.read_csv('cleaned_hr.csv')
df.head()

,Employee_Name,EmpID,MarriedID,MaritalStatusID,GenderID,EmpStatusID,DeptID,PerfScoreID,FromDiversityJobFairID,Salary,...,Hire_month_name,Hire_day_name,Absence_label,Absence_id,Absence_label_id,Col1,Col2,First Name,Middle Name,Last Name
0,Adinolfi Wilson K,10026,0,0,1,1,5,4,0,62506,...,July,Tuesday,Low Absences,0,0,Low,Absences,Wilson,K,Adinolfi
1,Ait Sidi Karthikeyan,10084,1,1,1,5,3,3,0,104437,...,March,Monday,High Absences,2,2,High,Absences,Karthikeyan,NaN,Ait Sidi
2,Akinkuolie Sarah,10196,1,1,0,5,5,3,0,64955,...,July,Tuesday,Low Absences,0,0,Low,Absences,Sarah,NaN,Akinkuolie
3,AlagbeTrina,10088,1,1,0,1,5,3,0,64991,...,January,Monday,High Absences,2,2,High,Absences,Trina,NaN,Alagbe
4,Anderson Carol,10069,0,2,0,5,5,3,0,50825,...,July,Monday,Low Absences,0,0,Low,Absences,Carol,NaN,Anderson


### Find total employees by each Employment Status.

In [4]:
status_employees=df.groupby('EmploymentStatus')['EmpID'].count() # we must always group by primary key
status_employees

EmploymentStatus
Active                    207
Terminated for Cause       16
Voluntarily Terminated     88
Name: EmpID, dtype: int64

In [5]:
status_employees=df.groupby('EmploymentStatus')['EmpID'].count().reset_index().rename(
    columns = {
        'EmpID': 'Total Employees',
        'EmploymentStatus': 'Employment Status'
    }
)
status_employees

,Employment Status,Total Employees
0,Active,207
1,Terminated for Cause,16
2,Voluntarily Terminated,88


In [6]:
status_employees=df.groupby('EmploymentStatus')['EmpID'].count().rename(
    'Total Employees').rename_axis('Employment Status').to_markdown(numalign='center',stralign='center')# number alignment and string
print(status_employees)

|   Employment Status    |  Total Employees  |
|:----------------------:|:-----------------:|
|         Active         |        207        |
|  Terminated for Cause  |        16         |
| Voluntarily Terminated |        88         |


In [7]:
emp_gender_count=df.groupby(['EmploymentStatus','Gender'])['EmpID'].count().reset_index()
emp_gender_count

,EmploymentStatus,Gender,EmpID
0,Active,F,116
1,Active,M,91
2,Terminated for Cause,F,9
3,Terminated for Cause,M,7
4,Voluntarily Terminated,F,51
5,Voluntarily Terminated,M,37


In [8]:
emp_gender_count=df.groupby(['EmploymentStatus','Gender'])['EmpID'].count().reset_index().rename(
    columns={
        'EmploymentStatus': 'Employment Status',
        'EmpID': 'Total Employees'
    }
)
emp_gender_count

,Employment Status,Gender,Total Employees
0,Active,F,116
1,Active,M,91
2,Terminated for Cause,F,9
3,Terminated for Cause,M,7
4,Voluntarily Terminated,F,51
5,Voluntarily Terminated,M,37


In [9]:
# Function for formatting the data in the columns
def currencyConverter(dataframe,prefix,*args):
    for col in args:
        dataframe[col]=dataframe[col].map(f'{prefix}''{:,.2f}'.format)

### Find total salary distributed and total employees in each department.

In [10]:
emp_sal= df.groupby('Department').agg(
    Total_Employees = ('EmpID','count'),
    Total_Salary = ('Salary','sum'),
    Average_Salary = ('Salary','mean'),
    Max_Salary= ('Salary','max'),
    Min_Salary = ('Salary','min')
).reset_index().sort_values(ascending=False, by='Total_Salary', ignore_index=True)

emp_sal.columns= emp_sal.columns.str.replace('_',' ',regex=False)

# emp_sal['Total Salary'] = emp_sal['Total Salary'].map('Rs {:,.2f}'.format)
currencyConverter(emp_sal,'Rs.','Total Salary','Average Salary','Max Salary','Min Salary')
emp_sal

,Department,Total Employees,Total Salary,Average Salary,Max Salary,Min Salary
0,Production,209,"Rs.12,530,291.00","Rs.59,953.55","Rs.170,500.00","Rs.45,046.00"
1,IT/IS,50,"Rs.4,853,232.00","Rs.97,064.64","Rs.220,450.00","Rs.50,178.00"
2,Sales,31,"Rs.2,140,899.00","Rs.69,061.26","Rs.180,000.00","Rs.55,875.00"
3,Software Engineering,11,"Rs.1,044,884.00","Rs.94,989.45","Rs.108,987.00","Rs.77,692.00"
4,Admin Offices,9,"Rs.646,127.00","Rs.71,791.89","Rs.106,367.00","Rs.49,920.00"
5,Executive Office,1,"Rs.250,000.00","Rs.250,000.00","Rs.250,000.00","Rs.250,000.00"


In [11]:
# Growth %
# ((current_value - baseline_value)/(baseline_value))*100

### Find total % of hiring rate of each year.

In [12]:
# pct_change() ,percentage change function is used to calculate growth% -> compares with the preceeding value.
def pctConverter(dataframe,suffix,*args):
    for col in args:
        dataframe[col]=dataframe[col].map(f'{suffix}''{:,.2f}'.format)
        
hire_rate = df.groupby('Hire_year')['EmpID'].count().reset_index()
hire_rate['Hire%']=(hire_rate['EmpID'].pct_change().fillna(0))*100
pctConverter(hire_rate,'%','Hire%')
hire_rate

,Hire_year,EmpID,Hire%
0,2006,1,%0.00
1,2007,2,%100.00
2,2008,3,%50.00
3,2009,7,%133.33
4,2010,9,%28.57
5,2011,83,%822.22
6,2012,45,%-45.78
7,2013,44,%-2.22
8,2014,60,%36.36
9,2015,36,%-40.00


## Total employees hired by each month

In [13]:
df.columns

Index(['Employee_Name', 'EmpID', 'MarriedID', 'MaritalStatusID', 'GenderID',
       'EmpStatusID', 'DeptID', 'PerfScoreID', 'FromDiversityJobFairID',
       'Salary', 'Termd', 'PositionID', 'Position', 'State', 'Zip', 'DOB',
       'Gender', 'MaritalDesc', 'CitizenDesc', 'HispanicLatino', 'RaceDesc',
       'DateofHire', 'DateofTermination', 'TermReason', 'EmploymentStatus',
       'Department', 'ManagerName', 'ManagerID', 'RecruitmentSource',
       'PerformanceScore', 'EngagementSurvey', 'EmpSatisfaction',
       'SpecialProjectsCount', 'LastPerformanceReview_Date', 'DaysLateLast30',
       'Absences', 'Hire_year', 'Hire_month', 'Hire_day', 'Hire_month_name',
       'Hire_day_name', 'Absence_label', 'Absence_id', 'Absence_label_id',
       'Col1', 'Col2', 'First Name', 'Middle Name', 'Last Name'],
      dtype='object')

In [14]:
month_emp=df.groupby(['Hire_month','Hire_month_name'])['EmpID'].count().reset_index().rename(
    columns={
        'Hire_month_name':'Hire Month',
        'EmpID':'Total Employee'
    }
)[['Hire Month','Total Employee']]
month_emp

,Hire Month,Total Employee
0,January,54
1,February,31
2,March,19
3,April,27
4,May,33
5,June,8
6,July,44
7,August,20
8,September,39
9,October,7


## Total employees hired by each day

In [15]:
# we form category so that the output prints in the order we set, we can't do like in month for days
weekdays=['Sunday','Monday','Tuesday','Wednesday','Thursday','Friday','Saturday']
df['Hire_day_name']=pd.Categorical(df['Hire_day_name'],categories=weekdays)
df['Hire_day_name'].unique()

['Tuesday', 'Monday', 'Thursday', 'Wednesday', 'Sunday', 'Friday', 'Saturday']
Categories (7, object): ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']

In [16]:
day_hired=df.groupby(['Hire_day_name'],observed=False)['EmpID'].count().reset_index()
# by observed False, if there is null value in any day then that is not shown in the output
day_hired

,Hire_day_name,EmpID
0,Sunday,6
1,Monday,259
2,Tuesday,19
3,Wednesday,9
4,Thursday,9
5,Friday,6
6,Saturday,3


## Key Perfromance Indicator (KPI)

In [17]:
# we use aggregate function to find KPI
# summarize data 
total_employees = df['EmpID'].count()
total_salary= df['Salary'].sum()
total_manager=df['ManagerID'].nunique()
total_departments=df['Department'].nunique()
total_projects=df['SpecialProjectsCount'].sum()

In [18]:
print(f'Total Employees: {total_employees}')
print(f'Total Salary: ${total_salary:,}')
print(f'Total Manager: {total_manager}')
print(f'Total Departments: {total_departments}')
print(f'Total Projects: {total_projects}')

Total Employees: 311
Total Salary: $21,465,433
Total Manager: 23
Total Departments: 6
Total Projects: 379
